# Comparison of calibrated uncertainty across canopy-height products

This notebook compares the **predictive uncertainty** of the local two-stage model and the published canopy-height maps used in the main article comparison:

- Our Model;
- Pauls et al. (Pa24);
- Lang et al. (L23);
- Tolan et al. (T24);

Potapov et al. (P21) remains registered for supplementary pairwise analyses but is excluded from the primary strict-common comparison because its valid spatial support substantially reduces the common GEDI sample in Ifran and Maamoura.

The comparison uses the same external uncertainty framework for every product. It does **not** compare heterogeneous native uncertainty layers. Instead, it constructs 90% split-conformal prediction intervals from GEDI residuals.

Two proposals are evaluated:

1. **Global split conformal (primary):** one product- and forest-specific residual quantile.
2. **Predicted-height-adaptive split conformal (secondary):** one residual quantile per predicted-height class, with a global fallback when calibration support is insufficient.

The main decision figure reports the trade-off between empirical TEST coverage (PICP90) and mean interval width (MPIW90). A useful product reaches the nominal 90% coverage with the narrowest interval.

> **Active lineage (2026-08-09).** This notebook uses the harmonised GEDI-anchored Phase 2 checkpoints. Training sequences contain at least one valid GEDI observation, while dense image-only sequences are reserved for wall-to-wall inference. Execute the notebook from the first cell; outputs from the former AOI-only Phase 2 lineage are not reused.


## Non-negotiable validity rules

The notebook enforces the following protocol:

- `VAL` GEDI observations calibrate the conformal quantiles.
- `TEST` GEDI observations are used only once, after calibration, to measure coverage and interval width.
- A GEDI shot appearing several times is reduced with the same **unique-nearest** rule used elsewhere in the project.
- Products are compared on the exact common set of valid GEDI footprints within each forest.
- The 25 m GEDI footprint is represented by an area-weighted mean of valid raster pixels; at least 80% footprint coverage is required.
- Prediction intervals are lower-bounded at 0 m because canopy height cannot be negative.

This is a conditional local calibration study. Because global products may have used GEDI observations from the wider region during their original training, the final support should be described as a **common local GEDI evaluation support**, not as universally independent of every product's training data.

In [ ]:
from pathlib import Path
import json, math, hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
from IPython.display import display
from rasterio.warp import transform
from rasterio.windows import Window, from_bounds
from shapely.geometry import Point

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
PRODUCT_ROOT = PROJECT / "CHM_Products_Comparison"
OUT = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "CHM_Uncertainty_Comparison"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"
RASTERS = OUT / "rasters"
for directory in (TABLES, FIGURES, RASTERS):
    directory.mkdir(parents=True, exist_ok=True)

RADIUS_M = 12.5
MIN_COVERAGE = 0.80
NOMINAL_COVERAGE = 0.90
MIN_ADAPTIVE_CALIBRATION = 100
RUN_PAIRING = True           # rebuild after any map, catalogue, checkpoint, or protocol change
RUN_RASTER_EXPORT = True   # optional and potentially slow
RANDOM_SEED = 42

OUR_PRODUCT = "Our Model"
PRODUCT_LABELS = {
    "Our Model": "Our Model",
    "Pauls 2020": "Pa24 (Pauls et al.)",
    "Lang 2020": "L23 (Lang et al.)",
    "Meta/Tolan 2023": "T24 (Tolan et al.)",
    "GFCH 2019": "P21 (Potapov et al.)",
}
PRODUCT_COLORS = {
    "Our Model": "#4C78A8", "Pauls 2020": "#E5C453",
    "Lang 2020": "#E17C54", "Meta/Tolan 2023": "#D88CAE",
    "GFCH 2019": "#72B7B2",
}

print("Outputs:", OUT)

## Product and forest registry

These paths reproduce the maps and catalogues used by `Chm_Comparison.ipynb`. Our Model is sampled from the annual Phase 2 map matching the GEDI acquisition year. Published global products are static maps and retain their native nominal dates.

In [ ]:
SITES = {
    "Ifran": {
        "key": "Ifran_6", "eval_max": 45.0,
        "adaptive_edges": [2, 5, 8, 10, 15, 20, 30, 40, 45],
        "catalog": Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Data\Dense\Ifran\Catalogs\final_catalog_C15_NATIVE"),
        "ours_template": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Dense/Ifran/Phase2/Y2020/Annual/Ifran_B4_C15_Phase2_Y2020_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
        "products": ["Our Model", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"],
    },
    "Maamoura": {
        "key": "Maamoura", "eval_max": 20.0,
        "adaptive_edges": [2, 5, 8, 10, 15, 20],
        "catalog": Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Data\Low_Sparsity\Maamoura\Temporal_Catalogs\T4_DENSE_2019_2025_C15"),
        "ours_template": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Low_Sparsity/Maamoura/Phase2/Y2019/Annual/Maamoura_B4_C15_Phase2_Y2019_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
        "products": ["Our Model", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"],
    },
    "Agadir": {
        "key": "Agadir", "eval_max": 20.0,
        "adaptive_edges": [2, 5, 8, 10, 15, 20],
        "catalog": Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Data\Sparse\Agadir\Catalogs\final_catalog"),
        "ours_template": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Sparse/Agadir/Phase2/Y2020/Annual/Agadir_B4_C15_Phase2_Y2020_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
        "products": ["Our Model", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"],
    },
}

def annual_path(template, year):
    import re
    text = str(template)
    years = re.findall(r"Y(20\d{2})", text)
    if not years:
        raise ValueError(f"Cannot identify year token in {template}")
    return Path(text.replace(f"Y{years[0]}", f"Y{int(year)}"))

for forest, cfg in SITES.items():
    root = PRODUCT_ROOT / cfg["key"]
    pauls_epsg = "EPSG32630" if forest == "Ifran" else "EPSG32629"
    pauls_root = PRODUCT_ROOT / "_PAULS_2020_VERIFIED_V1" / forest / "Pauls_et_al_2024_CHM_2020_10m/clean/mosaic"
    cfg["maps"] = {
        "Pauls 2020": pauls_root / f"{forest}__Pauls_et_al_2024_CHM_2020_10m__clean__{pauls_epsg}.tif",
        "Lang 2020": root / f"ETH_Lang_2020_CHM_10m/clean/mosaic/{cfg['key']}__ETH_Lang_2020_CHM_10m__clean__EPSG32630.tif",
        "Meta/Tolan 2023": root / f"Meta_WRI_Tolan_2023_CHM_resampled_10m/clean/mosaic/{cfg['key']}__Meta_WRI_Tolan_2023_CHM_resampled_10m__clean__EPSG32630.tif",
        "GFCH 2019": root / f"GFCH_Potapov_GLAD_2019_30m/clean/mosaic/{cfg['key']}__GFCH_Potapov_GLAD_2019_30m__clean__EPSG32630.tif",
    }

rows = []
for forest, cfg in SITES.items():
    rows.append({"forest": forest, "input": "GEDI catalogue", "path": cfg["catalog"] / "shot_catalog_step05.csv.gz"})
    for product in cfg["products"]:
        path = cfg["ours_template"] if product == OUR_PRODUCT else cfg["maps"][product]
        rows.append({"forest": forest, "input": product, "path": path})
preflight = pd.DataFrame(rows)
preflight["exists"] = preflight["path"].map(Path.is_file)
display(preflight)
if not preflight["exists"].all():
    raise FileNotFoundError("Missing required inputs:\n" + preflight.loc[~preflight.exists, "path"].astype(str).str.cat(sep="\n"))
final_registry = json.loads((PROJECT / "Source" / "Project" / "final_selected_phase2_models.json").read_text(encoding="utf-8"))["models"]
for forest, cfg in SITES.items():
    manifest_path = cfg["ours_template"].parent.parent / "QA" / "inference_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"{forest}: missing inference manifest: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = final_registry[forest.lower()]["checkpoint_sha256"]
    if manifest.get("phase2_sha256") != expected:
        raise RuntimeError(f"{forest}: stale annual map; rerun Inference.ipynb before conformal calibration")

## GEDI support and footprint sampling

For each split, the catalogue is filtered to the article's evaluation range. Repeated occurrences of one GEDI observation are resolved using the smallest absolute GEDI–Sentinel-2 temporal difference. Raster predictions are averaged over the GEDI footprint rather than read from a single pixel.

In [ ]:
def load_points(forest, cfg, split):
    shots = pd.read_csv(cfg["catalog"] / "shot_catalog_step05.csv.gz", low_memory=False)
    shots = shots[shots["split"].astype(str).str.lower().eq(split)].copy()
    shots["rh95"] = pd.to_numeric(shots["rh95"], errors="coerce")
    shots = shots[shots["rh95"].between(2.0, cfg["eval_max"])].copy()
    shots["gedi_year"] = pd.to_datetime(shots["aux_gedi_date"], errors="coerce").dt.year
    if {"aux_lon", "aux_lat"}.issubset(shots.columns):
        shots["lon"] = pd.to_numeric(shots["aux_lon"], errors="coerce")
        shots["lat"] = pd.to_numeric(shots["aux_lat"], errors="coerce")
    elif forest == "Ifran":
        coordinate_file = Path(r"E:\CHM\Ifran_6\DATA\GEDI\Output_Preproc_L2A\GEDI_QC_ACQ_clean.csv.gz")
        coordinates = pd.read_csv(coordinate_file, usecols=["shot_number", "lon", "lat"])
        coordinates["shot_number"] = pd.to_numeric(coordinates["shot_number"], errors="coerce").astype("Int64")
        shots["aux_shot_id"] = pd.to_numeric(shots["aux_shot_id"], errors="coerce").astype("Int64")
        shots = shots.merge(coordinates, left_on="aux_shot_id", right_on="shot_number", how="left", validate="many_to_one")
    else:
        raise KeyError(f"{forest}: catalogue has no aux_lon/aux_lat")
    shots = shots.dropna(subset=["lon", "lat", "rh95", "gedi_year"])
    shots = shots.sort_values(["aux_shot_uid", "aux_abs_temporal_delta_days", "aux_gedi_date"], kind="stable").drop_duplicates("aux_shot_uid")
    shots["shot_id"] = shots["aux_shot_uid"].astype(str)
    return shots[["shot_id", "rh95", "gedi_year", "lon", "lat"]]

def footprint_mean(src, x, y):
    footprint = Point(float(x), float(y)).buffer(RADIUS_M, quad_segs=24)
    raw = from_bounds(*footprint.bounds, transform=src.transform)
    c0, r0 = max(0, int(math.floor(raw.col_off))-1), max(0, int(math.floor(raw.row_off))-1)
    c1 = min(src.width, int(math.ceil(raw.col_off + raw.width))+1)
    r1 = min(src.height, int(math.ceil(raw.row_off + raw.height))+1)
    if c1 <= c0 or r1 <= r0: return np.nan, 0.0
    window = Window(c0, r0, c1-c0, r1-r0)
    arr = src.read(1, window=window, masked=True)
    values = np.ma.filled(arr, np.nan).astype(float)
    rr, cc = np.indices(values.shape); rr += r0; cc += c0
    a = src.transform
    xa, xb = a.c + cc*a.a, a.c + (cc+1)*a.a
    ya, yb = a.f + rr*a.e, a.f + (rr+1)*a.e
    pixels = shapely.box(np.minimum(xa, xb), np.minimum(ya, yb), np.maximum(xa, xb), np.maximum(ya, yb))
    areas = shapely.area(shapely.intersection(pixels, footprint))
    valid = (areas > 1e-9) & (~np.ma.getmaskarray(arr)) & np.isfinite(values) & (values >= 0) & (values <= 100)
    coverage = float(areas[valid].sum() / footprint.area)
    if coverage < MIN_COVERAGE or not valid.any(): return np.nan, coverage
    return float(np.sum(values[valid]*areas[valid]) / areas[valid].sum()), coverage

def sample_map(points, path):
    pred = np.full(len(points), np.nan); coverage = np.zeros(len(points))
    with rasterio.open(path) as src:
        xs, ys = transform("EPSG:4326", src.crs, points.lon.tolist(), points.lat.tolist())
        for i, (x, y) in enumerate(zip(xs, ys)):
            pred[i], coverage[i] = footprint_mean(src, x, y)
    return pred, coverage

## Construct the paired VAL and TEST tables

Each product is sampled independently, then the exact intersection of valid `shot_id` values is retained within each forest and split. This guarantees identical GEDI support across the displayed products.

The cell is resumable through a fingerprinted cache. Set `RUN_PAIRING=True` after changing any map, catalogue, or protocol setting.

In [ ]:
CACHE = OUT / "paired_VAL_TEST_common_GEDI_support.csv.gz"

def build_pairs():
    parts = []
    for forest, cfg in SITES.items():
        for split in ("val", "test"):
            points = load_points(forest, cfg, split)
            sampled = []
            for product in cfg["products"]:
                if product == OUR_PRODUCT:
                    pred = np.full(len(points), np.nan); cov = np.zeros(len(points))
                    for year, idx in points.groupby("gedi_year").groups.items():
                        path = annual_path(cfg["ours_template"], int(year))
                        if not path.is_file(): raise FileNotFoundError(path)
                        values, coverages = sample_map(points.loc[idx], path)
                        positions = points.index.get_indexer(idx)
                        pred[positions], cov[positions] = values, coverages
                else:
                    pred, cov = sample_map(points, cfg["maps"][product])
                frame = points.copy()
                frame["forest"], frame["split"], frame["product"] = forest, split.upper(), product
                frame["prediction"], frame["coverage"] = pred, cov
                sampled.append(frame[np.isfinite(frame.prediction)])
            stacked = pd.concat(sampled, ignore_index=True)
            id_sets = [set(stacked.loc[stacked["product"].eq(p), "shot_id"]) for p in cfg["products"]]
            common = set.intersection(*id_sets)
            if len(common) < 30: raise RuntimeError(f"{forest}/{split}: common support n={len(common)}")
            part = stacked[stacked.shot_id.isin(common)].copy()
            expected = len(common)
            assert part.groupby("product").shot_id.nunique().eq(expected).all()
            parts.append(part)
            print(f"[COMMON SUPPORT] {forest} {split.upper()}: n={expected:,}")
    result = pd.concat(parts, ignore_index=True)
    result["residual"] = result.rh95 - result.prediction
    result["abs_residual"] = result.residual.abs()
    return result

if RUN_PAIRING or not CACHE.is_file():
    paired = build_pairs()
    paired.to_csv(CACHE, index=False, compression="gzip")
else:
    paired = pd.read_csv(CACHE, low_memory=False)

overlap = set(paired.loc[paired.split.eq("VAL"), "shot_id"]) & set(paired.loc[paired.split.eq("TEST"), "shot_id"])
assert not overlap, f"VAL/TEST leakage: {len(overlap)} shared shot_id"
support = paired.groupby(["forest", "split", "product"]).shot_id.nunique().rename("n").reset_index()
display(support)

## Proposal 1 — Global split-conformal intervals

For product \(j\), the calibration residual is

\[
r_i^{(j)} = |y_i-\hat y_i^{(j)}|.
\]

The finite-sample conformal quantile is the `ceil((n+1) × 0.90)`-th ordered calibration residual. The TEST interval is

\[
[L,U]=[\max(0,\hat y-q_{0.90}),\hat y+q_{0.90}].
\]

This is the primary comparison because it is simple, distribution-free under exchangeability, and directly comparable across all CHMs.

In [ ]:
def conformal_quantile(values, coverage=NOMINAL_COVERAGE):
    values = np.sort(np.asarray(values, float)[np.isfinite(values)])
    n = len(values)
    if n == 0: return np.nan
    rank = min(n, int(np.ceil((n + 1) * coverage)))
    return float(values[rank - 1])

def interval_metrics(frame):
    covered = (frame.rh95 >= frame.lower) & (frame.rh95 <= frame.upper)
    width = frame.upper - frame.lower
    alpha = 1 - NOMINAL_COVERAGE
    score = width + (2/alpha)*(frame.lower-frame.rh95).clip(lower=0) + (2/alpha)*(frame.rh95-frame.upper).clip(lower=0)
    return {"n": len(frame), "PICP90": covered.mean(), "MPIW90_m": width.mean(), "median_width_m": width.median(), "mean_interval_score": score.mean()}

global_rows, global_test_parts = [], []
for (forest, product), group in paired.groupby(["forest", "product"], sort=False):
    val = group[group.split.eq("VAL")]
    test = group[group.split.eq("TEST")].copy()
    q = conformal_quantile(val.abs_residual)
    test["method"], test["q90"] = "Global", q
    test["lower"] = np.maximum(0, test.prediction - q)
    test["upper"] = test.prediction + q
    metrics = interval_metrics(test)
    global_rows.append({"forest": forest, "product": product, "q90_VAL_m": q, "n_VAL": len(val), **metrics})
    global_test_parts.append(test)

global_metrics = pd.DataFrame(global_rows)
global_test = pd.concat(global_test_parts, ignore_index=True)
global_metrics.to_csv(TABLES / "01_global_conformal_TEST_metrics.csv", index=False)
display(global_metrics.style.format({"q90_VAL_m":"{:.2f}", "PICP90":"{:.1%}", "MPIW90_m":"{:.2f}", "mean_interval_score":"{:.2f}"}))

## Proposal 2 — Predicted-height-adaptive intervals

Residual dispersion often changes with canopy height. The adaptive proposal estimates a separate VAL residual quantile within predicted-height classes. Binning uses **predicted**, not observed, height so that the rule remains available when producing a map. If a class contains fewer than 100 VAL observations, the product's global quantile is used.

This proposal is secondary: it can improve conditional coverage, but it adds complexity and should not replace the global result unless it provides a clear coverage–width benefit.

In [ ]:
adaptive_rows, adaptive_test_parts, q_rows = [], [], []
for (forest, product), group in paired.groupby(["forest", "product"], sort=False):
    edges = np.asarray(SITES[forest]["adaptive_edges"], float)
    val = group[group.split.eq("VAL")].copy(); test = group[group.split.eq("TEST")].copy()
    global_q = conformal_quantile(val.abs_residual)
    val["height_class"] = pd.cut(val.prediction, bins=edges, include_lowest=True, right=False)
    test["height_class"] = pd.cut(test.prediction, bins=edges, include_lowest=True, right=False)
    q_map = {}
    for cls, cls_val in val.groupby("height_class", observed=False):
        n = len(cls_val)
        q = conformal_quantile(cls_val.abs_residual) if n >= MIN_ADAPTIVE_CALIBRATION else global_q
        q_map[cls] = q
        q_rows.append({"forest":forest, "product":product, "height_class":str(cls), "n_VAL":n, "q90_m":q, "fallback_global":n < MIN_ADAPTIVE_CALIBRATION})
    test["q90"] = test.height_class.map(q_map).astype(float).fillna(global_q)
    test["method"] = "Adaptive"
    test["lower"] = np.maximum(0, test.prediction - test.q90)
    test["upper"] = test.prediction + test.q90
    metrics = interval_metrics(test)
    adaptive_rows.append({"forest":forest, "product":product, "global_fallback_q90_m":global_q, **metrics})
    adaptive_test_parts.append(test)

adaptive_metrics = pd.DataFrame(adaptive_rows)
adaptive_test = pd.concat(adaptive_test_parts, ignore_index=True)
adaptive_q = pd.DataFrame(q_rows)
adaptive_metrics.to_csv(TABLES / "02_adaptive_conformal_TEST_metrics.csv", index=False)
adaptive_q.to_csv(TABLES / "03_adaptive_VAL_quantiles_by_predicted_height.csv", index=False)
display(adaptive_metrics.style.format({"PICP90":"{:.1%}", "MPIW90_m":"{:.2f}", "mean_interval_score":"{:.2f}"}))

## Main decision figure — coverage versus sharpness

Coverage alone is insufficient: excessively wide intervals can reach 90% trivially. The figure therefore displays PICP90 against MPIW90. Preferred products lie near the 90% line and toward the left. Results are shown separately by forest because their height distributions and structural complexity differ.

In [ ]:
def coverage_width_figure(metrics, method, stem):
    forests = list(SITES)
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6), constrained_layout=True)
    for ax, forest in zip(axes, forests):
        data = metrics[metrics.forest.eq(forest)]
        for _, row in data.iterrows():
            ax.scatter(row.MPIW90_m, row.PICP90, s=90, color=PRODUCT_COLORS[row["product"]], edgecolor="black", linewidth=.5, label=PRODUCT_LABELS[row["product"]])
            ax.annotate(PRODUCT_LABELS[row["product"]].split()[0], (row.MPIW90_m, row.PICP90), xytext=(5,5), textcoords="offset points", fontsize=8)
        ax.axhline(.90, ls="--", color="black", lw=1)
        ax.set_title(forest, fontweight="bold")
        ax.set_xlabel("MPIW90 (m)")
        ax.set_ylim(max(0.5, data.PICP90.min()-.05), min(1.01, data.PICP90.max()+.05))
        ax.grid(alpha=.2)
    axes[0].set_ylabel("PICP90")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(.5, -.06))
    fig.suptitle(f"{method} split-conformal uncertainty on common GEDI TEST support", fontweight="bold")
    fig.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor="white")
    plt.show()

coverage_width_figure(global_metrics, "Global", "01_global_PICP90_vs_MPIW90")
coverage_width_figure(adaptive_metrics, "Predicted-height-adaptive", "02_adaptive_PICP90_vs_MPIW90")

## Global versus adaptive proposal

The next table quantifies whether adaptation changes TEST coverage, width and interval score. Adaptation should be retained only if it improves conditional behaviour without producing unjustifiably wide intervals or unstable class estimates.

In [ ]:
comparison = global_metrics.merge(adaptive_metrics, on=["forest","product"], suffixes=("_global","_adaptive"))
comparison["delta_PICP90"] = comparison.PICP90_adaptive - comparison.PICP90_global
comparison["delta_MPIW90_m"] = comparison.MPIW90_m_adaptive - comparison.MPIW90_m_global
comparison["delta_interval_score"] = comparison.mean_interval_score_adaptive - comparison.mean_interval_score_global
comparison.to_csv(TABLES / "04_global_vs_adaptive_comparison.csv", index=False)
display(comparison[["forest","product","PICP90_global","MPIW90_m_global","PICP90_adaptive","MPIW90_m_adaptive","delta_PICP90","delta_MPIW90_m","delta_interval_score"]].style.format({
    "PICP90_global":"{:.1%}","PICP90_adaptive":"{:.1%}","delta_PICP90":"{:+.1%}",
    "MPIW90_m_global":"{:.2f}","MPIW90_m_adaptive":"{:.2f}","delta_MPIW90_m":"{:+.2f}","delta_interval_score":"{:+.2f}"}))

## Coverage by observed-height class

This diagnostic detects under-coverage hidden by the overall PICP90, particularly for rare tall canopies. Observed GEDI height is appropriate here because this is an evaluation diagnostic, not a rule used to generate intervals.

In [ ]:
class_rows = []
for method_frame in (global_test, adaptive_test):
    for (forest, product, method), group in method_frame.groupby(["forest","product","method"]):
        edges = SITES[forest]["adaptive_edges"]
        group = group.copy(); group["GEDI_height_class"] = pd.cut(group.rh95, edges, include_lowest=True, right=False)
        for cls, part in group.groupby("GEDI_height_class", observed=False):
            if len(part) == 0: continue
            class_rows.append({"forest":forest,"product":product,"method":method,"GEDI_height_class":str(cls), **interval_metrics(part)})
class_metrics = pd.DataFrame(class_rows)
class_metrics.to_csv(TABLES / "05_TEST_coverage_by_GEDI_height_class.csv", index=False)
display(class_metrics)

## Optional uncertainty rasters

When enabled, this block generates product-specific PIW90 rasters from the adaptive VAL quantiles. These maps describe **locally calibrated predictive uncertainty conditional on the chosen calibration support**. They are not intrinsic uncertainty products supplied by the original map authors.

For a fair article figure, display the same geographic zones for every CHM. Choose zones before inspecting which product looks best; preferably reuse the qualitative comparison extents or use a fixed, reproducible stratified selection.

In [ ]:
def adaptive_q_raster(array, edges, table, global_q):
    result = np.full(array.shape, global_q, dtype=np.float32)
    for _, row in table.iterrows():
        text = str(row["height_class"])
        left_text, right_text = text[1:-1].split(",")
        left, right = float(left_text), float(right_text)
        result[(array >= left) & (array < right)] = float(row["q90_m"])
    return result

def write_uncertainty_raster(source_path, output_path, values, profile):
    profile = profile.copy()
    profile.update(dtype="float32", nodata=-9999, count=1, compress="deflate")
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(values.astype(np.float32), 1)

uncertainty_raster_rows = []
if RUN_RASTER_EXPORT:
    for forest, cfg in SITES.items():
        for product in cfg["products"]:
            source_path = cfg["ours_template"] if product == OUR_PRODUCT else cfg["maps"][product]
            table = adaptive_q[
                adaptive_q["forest"].eq(forest)
                & adaptive_q["product"].eq(product)
            ]
            global_q = float(global_metrics.loc[
                global_metrics["forest"].eq(forest)
                & global_metrics["product"].eq(product),
                "q90_VAL_m",
            ].iloc[0])
            with rasterio.open(source_path) as src:
                prediction = src.read(1, masked=True).filled(np.nan).astype(np.float32)
                valid = np.isfinite(prediction) & (prediction >= 0)
                adaptive_qmap = adaptive_q_raster(
                    prediction, cfg["adaptive_edges"], table, global_q
                )
                global_piw = np.where(
                    valid,
                    (prediction + global_q) - np.maximum(0, prediction - global_q),
                    -9999,
                )
                adaptive_piw = np.where(
                    valid,
                    (prediction + adaptive_qmap)
                    - np.maximum(0, prediction - adaptive_qmap),
                    -9999,
                )
                safe = product.replace("/", "_").replace(" ", "_")
                global_path = RASTERS / f"{forest}_{safe}_GLOBAL_PIW90.tif"
                adaptive_path = RASTERS / f"{forest}_{safe}_ADAPTIVE_PIW90.tif"
                write_uncertainty_raster(source_path, global_path, global_piw, src.profile)
                write_uncertainty_raster(source_path, adaptive_path, adaptive_piw, src.profile)
                uncertainty_raster_rows.extend([
                    {"forest": forest, "product": product, "method": "Global", "path": str(global_path)},
                    {"forest": forest, "product": product, "method": "Adaptive", "path": str(adaptive_path)},
                ])
                print(f"[SAVED] {forest} | {product} | global + adaptive PIW90")
    uncertainty_raster_manifest = pd.DataFrame(uncertainty_raster_rows)
    uncertainty_raster_manifest.to_csv(TABLES / "06_uncertainty_raster_manifest.csv", index=False)
    display(uncertainty_raster_manifest)
else:
    print("Raster export disabled. Set RUN_RASTER_EXPORT=True.")

## Publication-ready uncertainty-map comparison

The following figures display the **adaptive PIW90** maps for the same forest extent within each row. All products in a forest use one common colour scale, so colour differences are directly interpretable. Grey pixels denote unavailable product support.

The maps should be interpreted together with PICP90: a narrow interval is useful only when its empirical coverage remains close to 90%. The global method remains the primary quantitative result; adaptive maps provide the spatial diagnostic.

In [ ]:
from rasterio.enums import Resampling

def read_display_raster(path, max_pixels=700):
    with rasterio.open(path) as src:
        scale = max(src.width / max_pixels, src.height / max_pixels, 1.0)
        height = max(1, int(round(src.height / scale)))
        width = max(1, int(round(src.width / scale)))
        data = src.read(
            1, out_shape=(height, width),
            resampling=Resampling.nearest, masked=True,
        ).filled(np.nan).astype(np.float32)
    return data

if not RUN_RASTER_EXPORT:
    raise RuntimeError("Enable RUN_RASTER_EXPORT and execute the preceding cell first.")

for forest, cfg in SITES.items():
    products = cfg["products"]
    arrays, paths = [], []
    for product in products:
        safe = product.replace("/", "_").replace(" ", "_")
        raster_path = RASTERS / f"{forest}_{safe}_ADAPTIVE_PIW90.tif"
        if not raster_path.is_file():
            raise FileNotFoundError(raster_path)
        arrays.append(read_display_raster(raster_path))
        paths.append(raster_path)
    finite = np.concatenate([a[np.isfinite(a)] for a in arrays])
    vmax = float(np.nanpercentile(finite, 99))
    cmap = plt.get_cmap("viridis").copy(); cmap.set_bad("#d9d9d9")
    fig, axes = plt.subplots(1, len(products), figsize=(4.0*len(products), 4.25), constrained_layout=True)
    for ax, product, array in zip(axes, products, arrays):
        image = ax.imshow(array, cmap=cmap, vmin=0, vmax=vmax, interpolation="nearest")
        ax.set_title(PRODUCT_LABELS[product], fontsize=10, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
    colorbar = fig.colorbar(image, ax=axes, fraction=0.025, pad=0.015)
    colorbar.set_label("Adaptive PIW90 (m)")
    fig.suptitle(f"{forest}: locally calibrated canopy-height uncertainty", fontweight="bold")
    stem = FIGURES / f"03_{forest.lower()}_adaptive_PIW90_maps"
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    plt.show()

# One compact three-forest matrix for the article.
products = ["Our Model", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"]
fig, axes = plt.subplots(3, 4, figsize=(14.2, 10.2), constrained_layout=True)
for row, (forest, cfg) in enumerate(SITES.items()):
    row_arrays = []
    for product in products:
        safe = product.replace("/", "_").replace(" ", "_")
        row_arrays.append(read_display_raster(RASTERS / f"{forest}_{safe}_ADAPTIVE_PIW90.tif"))
    finite = np.concatenate([a[np.isfinite(a)] for a in row_arrays])
    vmax = float(np.nanpercentile(finite, 99))
    cmap = plt.get_cmap("viridis").copy(); cmap.set_bad("#d9d9d9")
    for col, (product, array) in enumerate(zip(products, row_arrays)):
        ax = axes[row, col]
        im = ax.imshow(array, cmap=cmap, vmin=0, vmax=vmax, interpolation="nearest")
        if row == 0: ax.set_title(PRODUCT_LABELS[product], fontsize=10, fontweight="bold")
        if col == 0: ax.set_ylabel(forest, fontsize=10, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
    cb = fig.colorbar(im, ax=axes[row, :], fraction=0.018, pad=0.010)
    cb.set_label("PIW90 (m)")
matrix_stem = FIGURES / "04_three_forests_adaptive_PIW90_matrix"
fig.savefig(matrix_stem.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(matrix_stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
plt.show()
print("Saved map figures in", FIGURES)

## Reproducibility manifest and paper-ready interpretation

The manifest records the protocol and all output paths. The article should report:

- calibration source (`VAL`) and evaluation source (`TEST`);
- common-support sample size for each forest;
- PICP90 and MPIW90 together;
- whether the displayed intervals are global or predicted-height-adaptive;
- that all products received the same external conformal calibration protocol;
- that uncertainty maps are conditional on local GEDI calibration and do not represent native uncertainty from the original products.

Do not select the best uncertainty method from TEST repeatedly. Use VAL diagnostics to choose the reporting protocol, then reserve TEST for the final comparison.

In [ ]:
manifest = {
    "notebook": "CHM_Conformal_Uncertainty_Comparison.ipynb",
    "nominal_coverage": NOMINAL_COVERAGE,
    "calibration_split": "VAL",
    "evaluation_split": "TEST",
    "support": "strict common valid GEDI footprints within forest and split",
    "footprint_radius_m": RADIUS_M,
    "minimum_valid_footprint_coverage": MIN_COVERAGE,
    "adaptive_min_calibration_n": MIN_ADAPTIVE_CALIBRATION,
    "lower_bound_m": 0.0,
    "products": PRODUCT_LABELS,
    "outputs": {"tables": str(TABLES), "figures": str(FIGURES), "rasters": str(RASTERS)},
}
(OUT / "reproducibility_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("Notebook complete. Review tables before enabling raster export.")